# Lesson 24: Interpretability & Explainable ML

*ML Academy — companion notebook*

Explain models globally and locally: standardised weights, permutation importance, partial dependence and ICE, SHAP values from scratch, LIME, and the traps.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-24.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Reading linear models properly

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y, names = data.data, data.target, data.feature_names      # target 1 = benign
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
lin = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(X_tr, y_tr)
coef = pd.Series(lin[-1].coef_[0], index=names)
print("largest standardised weights (positive pushes towards benign):")
print(coef.reindex(coef.abs().sort_values(ascending=False).index).head(5).round(3))

largest standardised weights (positive pushes towards benign):
worst texture          -0.543
worst radius           -0.529
worst perimeter        -0.496
worst concave points   -0.469
worst area             -0.463
dtype: float64


## 3. Permutation importance

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_tr, y_tr)
print("test accuracy:", round(rf.score(X_te, y_te), 3))
perm = permutation_importance(rf, X_te, y_te, n_repeats=30, random_state=0)
imp = pd.Series(perm.importances_mean, index=names).sort_values(ascending=False)
print("permutation importance on test data (drop in accuracy):")
print(imp.head(5).round(4))

test accuracy: 0.953


permutation importance on test data (drop in accuracy):
worst radius            0.0211
worst area              0.0177
worst concave points    0.0156
worst texture           0.0131
worst perimeter         0.0131
dtype: float64


## 4. Partial dependence and ICE plots

In [3]:
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import partial_dependence

d = load_diabetes()
Xd, yd = d.data, d.target
gbr = GradientBoostingRegressor(random_state=0).fit(Xd, yd)
bmi = list(d.feature_names).index("bmi")
pd_res = partial_dependence(gbr, Xd, [bmi], grid_resolution=6, kind="both")
print("bmi (scaled):        ", np.round(pd_res["grid_values"][0], 3))
print("average prediction:  ", np.round(pd_res["average"][0], 1))
ice = pd_res["individual"][0]
print("ICE spread at each bmi value (std across patients):", np.round(ice.std(axis=0), 1))

bmi (scaled):         [-0.067 -0.036 -0.005  0.025  0.056  0.087]
average prediction:   [130.9 129.1 141.7 176.4 177.  209.7]
ICE spread at each bmi value (std across patients): [45.2 40.6 49.  42.6 45.4 45.8]


## 5. SHAP values: fair credit for each feature

In [4]:
from itertools import combinations
from math import factorial

# Exact Shapley values for a model on 4 features, for one patient
feats = ["bmi", "bp", "s5", "age"]
idx = [list(d.feature_names).index(f) for f in feats]
X4 = Xd[:, idx]
model4 = GradientBoostingRegressor(random_state=0).fit(X4, yd)
background = X4[:100]                    # "absent" features are filled in from these patients
x = X4[200]

def value(subset):
    """Average prediction when the features in `subset` take x's values and the others vary as in the background."""
    Z = background.copy()
    Z[:, list(subset)] = x[list(subset)]
    return model4.predict(Z).mean()

p = len(feats); shap = np.zeros(p)
for j in range(p):
    others = [k for k in range(p) if k != j]
    for size in range(p):
        for S in combinations(others, size):
            weight = factorial(size) * factorial(p - size - 1) / factorial(p)
            shap[j] += weight * (value(S + (j,)) - value(S))

base, pred = value(()), model4.predict(x[None, :])[0]
for f, v in zip(feats, shap):
    print(f"{f:4}: {v:+7.2f}")
print(f"baseline {base:.2f}, plus the contributions ({shap.sum():.2f}), gives {base + shap.sum():.2f}")
print(f"the model's actual prediction: {pred:.2f}")

bmi :  -11.11
bp  :   +2.11
s5  :  -13.74
age :   +5.26
baseline 136.35, plus the contributions (-17.47), gives 118.87
the model's actual prediction: 118.87


## 7. Local surrogates (LIME)

In [5]:
from sklearn.linear_model import Ridge

# A LIME-style local explanation: fit a weighted linear model to the black box near one patient
rng = np.random.default_rng(0)
scale = X4.std(axis=0)
Z = x + rng.normal(0, 0.5, size=(2000, 4)) * scale          # perturbed copies of the patient
w = np.exp(-np.sum(((Z - x) / scale) ** 2, axis=1) / 2)        # nearer copies count more
local = Ridge(alpha=1e-3).fit((Z - x) / scale, model4.predict(Z), sample_weight=w)
print("local slopes (prediction change per +1 standard deviation):")
for f, c in zip(feats, local.coef_):
    print(f"{f:4}: {c:+7.2f}")

local slopes (prediction change per +1 standard deviation):
bmi :   +9.25
bp  :   +7.68
s5  :  +21.64
age :   +8.76


## 9. In Python: an explanation workflow

In [6]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance, partial_dependence

d = load_diabetes()
X_tr, X_te, y_tr, y_te = train_test_split(d.data, d.target, test_size=0.3, random_state=0)
model = GradientBoostingRegressor(random_state=0).fit(X_tr, y_tr)

# 1. Global: which features does the model rely on? (on test data)
perm = permutation_importance(model, X_te, y_te, n_repeats=20, random_state=0, scoring="r2")
top = np.argsort(perm.importances_mean)[::-1][:3]
print("top features:", [d.feature_names[i] for i in top], np.round(perm.importances_mean[top], 3))

# 2. Shape: how does the prediction change with the most important feature?
pdr = partial_dependence(model, X_tr, [top[0]], grid_resolution=5, method="brute")
print("partial dependence:", np.round(pdr["average"][0], 1))

top features: ['s5', 'bmi', 'bp'] [0.221 0.181 0.043]
partial dependence: [103.4 129.  155.  186.5 194.5]


## 12. Going deeper: when features are correlated

In [7]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# Two copies of the same signal: how do importance methods share the credit?
rng = np.random.default_rng(0)
n = 2000
a = rng.normal(size=n); b = a + rng.normal(0, 0.05, n)        # b is almost a copy of a
c = rng.normal(size=n)
X = np.column_stack([a, b, c]); y = 3 * a + 1 * c + rng.normal(0, 0.5, n)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
rf = RandomForestRegressor(n_estimators=200, random_state=0).fit(X_tr, y_tr)
perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0, scoring="r2").importances_mean
print("permutation importance (a, b, c):", perm.round(3))

# Drop-column importance: retrain without each feature and measure the loss
full = rf.score(X_te, y_te)
drop = [full - RandomForestRegressor(n_estimators=200, random_state=0).fit(np.delete(X_tr, j, 1), y_tr).score(np.delete(X_te, j, 1), y_te) for j in range(3)]
print("drop-column importance (a, b, c):", np.round(drop, 3))

permutation importance (a, b, c): [1.045 0.129 0.172]


drop-column importance (a, b, c): [0.004 0.    0.117]


In [8]:
# For a linear model the exact SHAP value of feature j is w_j * (x_j - mean_j); averaging |SHAP| gives a global ranking
lin = LinearRegression().fit(X_tr, y_tr)
shap_vals = lin.coef_ * (X_te - X_tr.mean(axis=0))
print("weights (a, b, c):            ", lin.coef_.round(2))
print("mean |SHAP| per feature:      ", np.abs(shap_vals).mean(axis=0).round(3))
print("check, first row: baseline + contributions", round(lin.predict(X_tr).mean() + shap_vals[0].sum(), 4), "vs prediction", round(lin.predict(X_te[:1])[0], 4))

weights (a, b, c):             [2.75 0.24 1.02]
mean |SHAP| per feature:       [2.31  0.202 0.791]
check, first row: baseline + contributions 3.9815 vs prediction 3.9815


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
cols = ["size_m2", "dist_km", "age_yrs"]
Xs = StandardScaler().fit_transform(df[cols])
lr = LinearRegression().fit(Xs, df["price_lakh"])
base = lr.intercept_                                  # average price (features are centred)
contrib = Xs * lr.coef_                               # for a linear model, SHAP = weight × centred value
h = 4                                                 # house id 5
print("baseline", round(base, 1), "| contributions:", {c: float(v) for c, v in zip(cols, contrib[h].round(1))})
print("prediction", round(base + contrib[h].sum(), 1), "| actual", df.price_lakh[h])

baseline 72.1 | contributions: {'size_m2': 2.4, 'dist_km': -0.2, 'age_yrs': 8.6}
prediction 82.9 | actual 88


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Permutation importance by hand

Shuffle one column and measure how much the score drops.

In [11]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.normal(size=(500, 2)); y = 3 * X[:, 0] + 0.1 * rng.normal(size=500)
w = np.array([3.0, 0.0])
score = lambda X: -np.mean((X @ w - y) ** 2)
Xp = X.copy(); Xp[:, 0] = rng.permutation(Xp[:, 0])
drop = score(X)  # ← replace: subtract the shuffled score
result = round(float(drop), 1)
check(result, 17.3, hint='Original score minus the score with feature 0 shuffled.')

❌ Not yet. Hint: Original score minus the score with feature 0 shuffled.


### Challenge 2: Shapley values for two features

For f(a, b) = 2a + 4ab at (1, 1) with baseline (0, 0), compute a's Shapley value as the average of its two marginal contributions.

In [12]:
f = lambda a, b: 2 * a + 4 * a * b
phi_a = f(1, 1)  # ← replace with the average marginal contribution
check(phi_a, 4.0, hint='a joins first (adds f(1,0) − f(0,0)) or second (adds f(1,1) − f(0,1)); average the two.')

❌ Not yet. Hint: a joins first (adds f(1,0) − f(0,0)) or second (adds f(1,1) − f(0,1)); average the two.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Permutation importance by hand** — the missing line:

```python
drop = score(X) - score(Xp)
```

**2. Shapley values for two features** — the missing line:

```python
phi_a = ((f(1, 0) - f(0, 0)) + (f(1, 1) - f(0, 1))) / 2
```